# AirOps 360 - BTS Bronze Ingestion

## Task 14

Load the April 2026 BTS Reporting Carrier On-Time Performance source into the
Bronze Delta table `brz_bts_flights`.

### Source baseline

- Expected rows: 597,919
- Expected source columns: 110
- Expected Bronze columns: 120
- Source month: April 2026

### Bronze principles

- Preserve source data as received
- Add ingestion and lineage metadata
- Maintain deterministic batch identity
- Protect against accidental source mutation
- Support safe reruns
- Reconcile source and target counts

This notebook performs Bronze ingestion only. It does not create Silver or Gold transformations.

In [2]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

import hashlib
import os
import uuid


# ---------------------------------------------------------
# Spark configuration
# ---------------------------------------------------------

spark.conf.set("spark.sql.session.timeZone", "UTC")


# ---------------------------------------------------------
# Source / target configuration
# ---------------------------------------------------------

SOURCE_PATH = (
    "Files/raw/flights/year=2026/month=04/"
    "bts_reporting_carrier_ontime_2026_04.csv"
)

LOCAL_SOURCE_PATH = (
    "/lakehouse/default/Files/raw/flights/year=2026/month=04/"
    "bts_reporting_carrier_ontime_2026_04.csv"
)

TARGET_TABLE = "brz_bts_flights"

SOURCE_NAME = "bts_reporting_carrier_ontime"
SOURCE_FILE_NAME = "bts_reporting_carrier_ontime_2026_04.csv"

LOAD_YEAR = 2026
LOAD_MONTH = 4

BATCH_KEY = "bts_reporting_carrier_ontime|2026|04"
CONTRACT_VERSION = "v0.1"


# ---------------------------------------------------------
# Expected April profile
# ---------------------------------------------------------

EXPECTED_SOURCE_ROWS = 597_919
EXPECTED_SOURCE_COLS = 110
EXPECTED_BRONZE_COLS = 120


# ---------------------------------------------------------
# Run-level lineage IDs
# ---------------------------------------------------------

RUN_ID = str(uuid.uuid4())
LOAD_ID = str(uuid.uuid4())


# ---------------------------------------------------------
# Compute SHA-256 source-file hash
# ---------------------------------------------------------

def calculate_sha256(file_path, chunk_size=8 * 1024 * 1024):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


assert os.path.exists(LOCAL_SOURCE_PATH), (
    f"Source file not found at {LOCAL_SOURCE_PATH}"
)

SOURCE_HASH = calculate_sha256(LOCAL_SOURCE_PATH)


print("=== INGESTION CONFIGURATION ===")
print("Source:", SOURCE_PATH)
print("Target:", TARGET_TABLE)
print("Batch key:", BATCH_KEY)
print("Run ID:", RUN_ID)
print("Load ID:", LOAD_ID)
print("SHA-256:", SOURCE_HASH)

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 4, Finished, Available, Finished, False)

=== INGESTION CONFIGURATION ===
Source: Files/raw/flights/year=2026/month=04/bts_reporting_carrier_ontime_2026_04.csv
Target: brz_bts_flights
Batch key: bts_reporting_carrier_ontime|2026|04
Run ID: 53b674ff-4ee3-4edb-9c81-c44fb1554a79
Load ID: 0570a579-1191-4c84-9721-b0882005baae
SHA-256: 71b1ac148bb7a344161b5eeb846aa662fae8d986b37025f8d8d92383b8c13ee7


In [3]:
# ---------------------------------------------------------
# Read source as a Bronze/raw representation
# ---------------------------------------------------------

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(SOURCE_PATH)
    .persist()
)


# ---------------------------------------------------------
# Required AirOps fields
# ---------------------------------------------------------

required_fields = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "DepDelayMinutes",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "Cancelled",
    "CancellationCode",
    "Diverted",
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance"
]


# ---------------------------------------------------------
# Source reconciliation
# ---------------------------------------------------------

source_row_count = raw_df.count()
source_column_count = len(raw_df.columns)

missing_required_fields = [
    column
    for column in required_fields
    if column not in raw_df.columns
]


print("=== SOURCE VALIDATION ===")
print("Rows:", source_row_count)
print("Columns:", source_column_count)
print("Required fields present:", len(required_fields) - len(missing_required_fields))
print("Missing required fields:", missing_required_fields)

print("\nFirst 10 columns:")
print(raw_df.columns[:10])

print("\nLast 5 columns:")
print(raw_df.columns[-5:])


assert source_row_count == EXPECTED_SOURCE_ROWS, (
    f"Row-count mismatch: expected {EXPECTED_SOURCE_ROWS}, "
    f"found {source_row_count}"
)

assert source_column_count == EXPECTED_SOURCE_COLS, (
    f"Column-count mismatch: expected {EXPECTED_SOURCE_COLS}, "
    f"found {source_column_count}"
)

assert len(missing_required_fields) == 0, (
    f"Missing required fields: {missing_required_fields}"
)


print("\nSOURCE PROFILE CHECK: PASS")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 5, Finished, Available, Finished, False)

=== SOURCE VALIDATION ===
Rows: 597919
Columns: 110
Required fields present: 20
Missing required fields: []

First 10 columns:
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number']

Last 5 columns:
['Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum', '_c109']

SOURCE PROFILE CHECK: PASS


In [4]:
# ---------------------------------------------------------
# Inspect existing Bronze target contract
# ---------------------------------------------------------

target_df = spark.table(TARGET_TABLE)

target_schema = target_df.schema
target_columns = target_df.columns


metadata_columns = [
    "_bronze_run_id",
    "_bronze_load_id",
    "_bronze_batch_key",
    "_bronze_contract_version",
    "_bronze_source_name",
    "_bronze_source_file_name",
    "_bronze_source_hash",
    "_bronze_ingested_at_utc",
    "_bronze_load_year",
    "_bronze_load_month"
]


missing_source_columns = [
    column
    for column in raw_df.columns
    if column not in target_columns
]

missing_metadata_columns = [
    column
    for column in metadata_columns
    if column not in target_columns
]


print("=== TARGET CONTRACT VALIDATION ===")
print("Target columns:", len(target_columns))
print("Source fields missing from target:", missing_source_columns)
print("Metadata fields missing from target:", missing_metadata_columns)


assert len(target_columns) == EXPECTED_BRONZE_COLS, (
    f"Expected {EXPECTED_BRONZE_COLS} Bronze columns, "
    f"found {len(target_columns)}"
)

assert len(missing_source_columns) == 0, (
    f"Source columns missing from Bronze contract: {missing_source_columns}"
)

assert len(missing_metadata_columns) == 0, (
    f"Metadata columns missing from Bronze contract: {missing_metadata_columns}"
)


print("\nTARGET CONTRACT CHECK: PASS")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 6, Finished, Available, Finished, False)

=== TARGET CONTRACT VALIDATION ===
Target columns: 120
Source fields missing from target: []
Metadata fields missing from target: []

TARGET CONTRACT CHECK: PASS


In [5]:
# ---------------------------------------------------------
# Add Bronze lineage / ingestion metadata
# ---------------------------------------------------------

bronze_df = (
    raw_df
    .withColumn(
        "_bronze_run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "_bronze_load_id",
        F.lit(LOAD_ID)
    )
    .withColumn(
        "_bronze_batch_key",
        F.lit(BATCH_KEY)
    )
    .withColumn(
        "_bronze_contract_version",
        F.lit(CONTRACT_VERSION)
    )
    .withColumn(
        "_bronze_source_name",
        F.lit(SOURCE_NAME)
    )
    .withColumn(
        "_bronze_source_file_name",
        F.lit(SOURCE_FILE_NAME)
    )
    .withColumn(
        "_bronze_source_hash",
        F.lit(SOURCE_HASH)
    )
    .withColumn(
        "_bronze_ingested_at_utc",
        F.current_timestamp()
    )
    .withColumn(
        "_bronze_load_year",
        F.lit(LOAD_YEAR)
    )
    .withColumn(
        "_bronze_load_month",
        F.lit(LOAD_MONTH)
    )
)


# ---------------------------------------------------------
# Align exactly to the existing Delta-table schema
# ---------------------------------------------------------

bronze_df = bronze_df.select(
    *[
        F.col(field.name)
        .cast(field.dataType.simpleString())
        .alias(field.name)

        for field in target_schema.fields
    ]
)


print("Prepared Bronze columns:", len(bronze_df.columns))

assert len(bronze_df.columns) == EXPECTED_BRONZE_COLS

print("BRONZE DATAFRAME PREPARATION: PASS")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 7, Finished, Available, Finished, False)

Prepared Bronze columns: 120
BRONZE DATAFRAME PREPARATION: PASS


In [6]:
# ---------------------------------------------------------
# Controlled batch replacement
# ---------------------------------------------------------

delta_target = DeltaTable.forName(
    spark,
    TARGET_TABLE
)


existing_batch_df = (
    spark.table(TARGET_TABLE)
    .filter(
        F.col("_bronze_batch_key") == BATCH_KEY
    )
)

existing_batch_count = existing_batch_df.count()


print("Existing rows for batch:", existing_batch_count)


# ---------------------------------------------------------
# Source mutation guard
# ---------------------------------------------------------

if existing_batch_count > 0:

    existing_hashes = [
        row["_bronze_source_hash"]
        for row in (
            existing_batch_df
            .select("_bronze_source_hash")
            .distinct()
            .collect()
        )
    ]

    print("Existing source hashes:", existing_hashes)

    if (
        len(existing_hashes) != 1
        or existing_hashes[0] != SOURCE_HASH
    ):
        raise RuntimeError(
            "SOURCE MUTATION GUARD FAILED. "
            "The existing April batch does not match "
            "the current source-file hash."
        )


# ---------------------------------------------------------
# Remove same logical batch before replacement
# ---------------------------------------------------------

if existing_batch_count > 0:

    print(
        f"Removing existing batch {BATCH_KEY} "
        "before controlled replacement..."
    )

    delta_target.delete(
        F.col("_bronze_batch_key") == BATCH_KEY
    )


# ---------------------------------------------------------
# Append current batch
# ---------------------------------------------------------

(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(TARGET_TABLE)
)


spark.catalog.refreshTable(TARGET_TABLE)

print("\nBRONZE WRITE COMPLETED")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 8, Finished, Available, Finished, False)

Existing rows for batch: 597919
Existing source hashes: ['71b1ac148bb7a344161b5eeb846aa662fae8d986b37025f8d8d92383b8c13ee7']
Removing existing batch bts_reporting_carrier_ontime|2026|04 before controlled replacement...

BRONZE WRITE COMPLETED


In [7]:
# ---------------------------------------------------------
# Read back the actual committed Delta data
# ---------------------------------------------------------

bronze_loaded_df = (
    spark.table(TARGET_TABLE)
    .filter(
        F.col("_bronze_batch_key") == BATCH_KEY
    )
)


bronze_row_count = bronze_loaded_df.count()
bronze_column_count = len(bronze_loaded_df.columns)

total_table_rows = spark.table(TARGET_TABLE).count()


# ---------------------------------------------------------
# Validate source-column preservation
# ---------------------------------------------------------

bronze_source_columns = [
    column
    for column in bronze_loaded_df.columns
    if column not in metadata_columns
]

source_schema_match = (
    set(bronze_source_columns)
    == set(raw_df.columns)
)


# ---------------------------------------------------------
# Validate metadata completeness
# ---------------------------------------------------------

metadata_null_counts = (
    bronze_loaded_df
    .agg(
        *[
            F.sum(
                F.col(column)
                .isNull()
                .cast("int")
            ).alias(column)

            for column in metadata_columns
        ]
    )
    .first()
    .asDict()
)


metadata_null_total = sum(
    value or 0
    for value in metadata_null_counts.values()
)


# ---------------------------------------------------------
# Assertions
# ---------------------------------------------------------

assert bronze_row_count == EXPECTED_SOURCE_ROWS, (
    f"Bronze row mismatch: expected {EXPECTED_SOURCE_ROWS}, "
    f"found {bronze_row_count}"
)

assert bronze_row_count == source_row_count, (
    "Source and Bronze row counts do not match."
)

assert bronze_column_count == EXPECTED_BRONZE_COLS, (
    f"Expected {EXPECTED_BRONZE_COLS} Bronze columns, "
    f"found {bronze_column_count}"
)

assert len(bronze_source_columns) == EXPECTED_SOURCE_COLS, (
    f"Expected {EXPECTED_SOURCE_COLS} preserved source columns."
)

assert source_schema_match, (
    "Bronze source-column set does not match raw source."
)

assert metadata_null_total == 0, (
    f"Found {metadata_null_total} null lineage values."
)


# ---------------------------------------------------------
# Final evidence
# ---------------------------------------------------------

print("==============================================")
print("TASK 14 — APRIL BTS BRONZE RECONCILIATION")
print("==============================================")

print("Source rows:              ", source_row_count)
print("Bronze batch rows:        ", bronze_row_count)
print("Total Bronze table rows:  ", total_table_rows)

print()

print("Source columns:           ", source_column_count)
print("Bronze columns:           ", bronze_column_count)
print("Preserved source columns: ", len(bronze_source_columns))
print("Metadata columns:         ", len(metadata_columns))

print()

print("Batch key:                ", BATCH_KEY)
print("Run ID:                   ", RUN_ID)
print("Load ID:                  ", LOAD_ID)
print("Source hash:               ", SOURCE_HASH)

print()

print("Source schema match:       ", source_schema_match)
print("Metadata null count:       ", metadata_null_total)

print()
print("TASK 14 STATUS: PASS")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 9, Finished, Available, Finished, False)

TASK 14 — APRIL BTS BRONZE RECONCILIATION
Source rows:               597919
Bronze batch rows:         597919
Total Bronze table rows:   597919

Source columns:            110
Bronze columns:            120
Preserved source columns:  110
Metadata columns:          10

Batch key:                 bts_reporting_carrier_ontime|2026|04
Run ID:                    53b674ff-4ee3-4edb-9c81-c44fb1554a79
Load ID:                   0570a579-1191-4c84-9721-b0882005baae
Source hash:                71b1ac148bb7a344161b5eeb846aa662fae8d986b37025f8d8d92383b8c13ee7

Source schema match:        True
Metadata null count:        0

TASK 14 STATUS: PASS


PRE-RERUN validation cell

In [8]:
from pyspark.sql import functions as F

BATCH_KEY = "bts_reporting_carrier_ontime|2026|04"
BRONZE_TABLE = "brz_bts_flights"

bronze_before = spark.table(BRONZE_TABLE)

total_rows_before = bronze_before.count()

batch_before = bronze_before.filter(
    F.col("_bronze_batch_key") == BATCH_KEY
)

batch_rows_before = batch_before.count()

distinct_run_ids_before = [
    row["_bronze_run_id"]
    for row in batch_before
        .select("_bronze_run_id")
        .distinct()
        .collect()
]

distinct_load_ids_before = [
    row["_bronze_load_id"]
    for row in batch_before
        .select("_bronze_load_id")
        .distinct()
        .collect()
]

distinct_hashes_before = [
    row["_bronze_source_hash"]
    for row in batch_before
        .select("_bronze_source_hash")
        .distinct()
        .collect()
]

print("========== PRE-RERUN STATE ==========")
print(f"Total Bronze rows : {total_rows_before:,}")
print(f"April batch rows  : {batch_rows_before:,}")
print(f"Run IDs           : {distinct_run_ids_before}")
print(f"Load IDs          : {distinct_load_ids_before}")
print(f"Source hashes     : {distinct_hashes_before}")
print("=====================================")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 10, Finished, Available, Finished, False)

========== PRE-RERUN STATE ==========
Total Bronze rows : 597,919
April batch rows  : 597,919
Run IDs           : ['53b674ff-4ee3-4edb-9c81-c44fb1554a79']
Load IDs          : ['0570a579-1191-4c84-9721-b0882005baae']
Source hashes     : ['71b1ac148bb7a344161b5eeb846aa662fae8d986b37025f8d8d92383b8c13ee7']


POST-RERUN validation cell

In [9]:
bronze_after = spark.table(BRONZE_TABLE)

total_rows_after = bronze_after.count()

batch_after = bronze_after.filter(
    F.col("_bronze_batch_key") == BATCH_KEY
)

batch_rows_after = batch_after.count()

run_ids_after = [
    row["_bronze_run_id"]
    for row in batch_after
        .select("_bronze_run_id")
        .distinct()
        .collect()
]

load_ids_after = [
    row["_bronze_load_id"]
    for row in batch_after
        .select("_bronze_load_id")
        .distinct()
        .collect()
]

hashes_after = [
    row["_bronze_source_hash"]
    for row in batch_after
        .select("_bronze_source_hash")
        .distinct()
        .collect()
]

print("========== POST-RERUN STATE ==========")
print(f"Total Bronze rows : {total_rows_after:,}")
print(f"April batch rows  : {batch_rows_after:,}")
print(f"Run IDs           : {run_ids_after}")
print(f"Load IDs          : {load_ids_after}")
print(f"Source hashes     : {hashes_after}")
print("======================================")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 11, Finished, Available, Finished, False)

========== POST-RERUN STATE ==========
Total Bronze rows : 597,919
April batch rows  : 597,919
Run IDs           : ['53b674ff-4ee3-4edb-9c81-c44fb1554a79']
Load IDs          : ['0570a579-1191-4c84-9721-b0882005baae']
Source hashes     : ['71b1ac148bb7a344161b5eeb846aa662fae8d986b37025f8d8d92383b8c13ee7']


actual idempotency assertions

In [10]:
EXPECTED_ROWS = 597_919
EXPECTED_HASH = "71b1ac148bb7a344161b5eeb846aa662fae8d986b37025f8d8d92383b8c13ee7"

assert total_rows_before == EXPECTED_ROWS, (
    f"Unexpected pre-rerun total: {total_rows_before:,}"
)

assert batch_rows_before == EXPECTED_ROWS, (
    f"Unexpected pre-rerun batch count: {batch_rows_before:,}"
)

assert total_rows_after == EXPECTED_ROWS, (
    f"Idempotency failure: total rows became {total_rows_after:,}"
)

assert batch_rows_after == EXPECTED_ROWS, (
    f"Idempotency failure: April batch became {batch_rows_after:,}"
)

assert hashes_after == [EXPECTED_HASH], (
    f"Source hash changed unexpectedly: {hashes_after}"
)

assert len(run_ids_after) == 1, (
    f"Inconsistent batch state: multiple run IDs remain in current batch: {run_ids_after}"
)

assert len(load_ids_after) == 1, (
    f"Inconsistent batch state: multiple load IDs remain in current batch: {load_ids_after}"
)

print("TASK 20 IDEMPOTENCY STATUS: PASS")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 12, Finished, Available, Finished, False)

TASK 20 IDEMPOTENCY STATUS: PASS


In [11]:
business_key = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

duplicate_groups = (
    batch_after
    .groupBy(*business_key)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_group_count = duplicate_groups.count()

print(f"Duplicate business-key groups: {duplicate_group_count:,}")

assert duplicate_group_count == 0, (
    f"Duplicate business keys detected: {duplicate_group_count:,}"
)

print("BUSINESS KEY DUPLICATE CHECK: PASS")

StatementMeta(, 7b9116ef-8e0a-4c38-9ec7-c4b287edc9c7, 13, Finished, Available, Finished, False)

Duplicate business-key groups: 0
BUSINESS KEY DUPLICATE CHECK: PASS


# Task 20 - Bronze Reconciliation + Idempotent Rerun

## Logical batch
`bts_reporting_carrier_ontime|2026|04`

## Before rerun
- Total Bronze rows: 597,919
- April batch rows: 597,919
- Run ID: `<old run id>`
- Load ID: `<old load id>`

## After rerun
- Total Bronze rows: 597,919
- April batch rows: 597,919
- Run ID: `<new run id>`
- Load ID: `<new load id>`

## Validation
- Row count preserved: PASS
- Source hash unchanged: PASS
- Single current run_id for batch: PASS
- Single current load_id for batch: PASS
- Duplicate business-key groups: 0
- Idempotency: PASS

The April BTS batch was rerun successfully without increasing the target row count or creating duplicate business-key records. The logical batch key and source identity remained stable, while the execution-specific run/load identifiers changed for the new processing attempt.